In [1]:
from pyspark.sql import SparkSession

In [2]:
BUCKET_NAME = "healthcare-tungpd20"

CPT_BUCKET_PATH = f"gs://{BUCKET_NAME}/landing/cptcodes/*.csv"

BQ_PROJECT = "project-8bf7907a-2104-49b0-99f"
BQ_DATASET = "bronze_dataset"
BQ_TABLE = f"{BQ_PROJECT}.{BQ_DATASET}.cpt_codes"

TEMP_GCS_BUCKET = f"{BUCKET_NAME}/temp/"


In [3]:
cptcodes_df = spark.read.csv(
    CPT_BUCKET_PATH,
    header=True
)


In [4]:
cptcodes_df.printSchema()

root
 |-- procedure code category: string (nullable = true)
 |-- cpt codes: string (nullable = true)
 |-- procedure code descriptions: string (nullable = true)
 |-- code status: string (nullable = true)



In [5]:
# replace spaces with underscore
for col in cptcodes_df.columns:
    new_col = col.replace(" ", "_").lower()
    cptcodes_df = cptcodes_df.withColumnRenamed(col, new_col)

In [6]:
# write to bigquery
(cptcodes_df.write
            .format("bigquery")
            .option("table", BQ_TABLE)
            .option("temporaryGcsBucket", TEMP_GCS_BUCKET)
            .mode("overwrite")
            .save())